# Простое обучение с подкреплением в TensorFlow


Простой пример построения агента на основе policy gradient, который может решить задачу CartPole. Эта реализация обобщается на случаи, где действий больше двух.



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import gym
import matplotlib.pyplot as plt
from collections import deque


In [4]:
env = gym.make('CartPole-v0')

[2026-05-09 18:45:39,894] Making new env: CartPole-v0


### Агент на основе политики

In [ ]:
gamma = 0.99

def discount_rewards(r):
    """Принимает одномерный массив наград и считает дисконтированную награду."""
    discounted_r = np.zeros_like(r)
    running_add = 0
    for t in reversed(range(0, r.size)):
        running_add = running_add * gamma + r[t]
        discounted_r[t] = running_add
    return discounted_r

In [ ]:
class Agent:
    def __init__(self, lr, state_size, action_size, hidden_size):
        self.network = nn.Sequential(
            nn.Linear(state_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, action_size),
            nn.Softmax(dim=-1)
        )
        self.optimizer = optim.Adam(self.network.parameters(), lr=lr)
        
        #Буфер для хранения данных эпизода
        self.states = []
        self.actions = []
        self.rewards = []
        self.log_probs = []
        

        self.grad_buffer = None
        
    def get_action(self, state):
        """Выбор действия"""
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        action_probs = self.network(state_tensor)
        
        # Создаем распределение и сэмплируем
        dist = torch.distributions.Categorical(action_probs)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        
        # Сохраняем для обучения
        self.states.append(state)
        self.actions.append(action.item())
        self.log_probs.append(log_prob)
        
        return action.item()
    
    def store_reward(self, reward):
        """Сохраняем награду"""
        self.rewards.append(reward)
    
    def learn(self):
        """Обучение на эпизоде"""
        # 1. Превращаем списки в тензоры
        states = torch.FloatTensor(np.vstack(self.states))
        actions = torch.LongTensor(self.actions)
        rewards = np.array(self.rewards, dtype=np.float32)
        
        # 2. Дисконтируем награды
        discounted_rewards = discount_rewards(rewards)
        discounted_rewards = torch.FloatTensor(discounted_rewards)
        
        # 3. Считаем логарифмы вероятностей выбранных действий
        action_probs = self.network(states)
        # Вытаскиваем вероятности для выбранных действий
        responsible_outputs = action_probs.gather(1, actions.unsqueeze(1)).squeeze(1)
        log_responsible_outputs = torch.log(responsible_outputs)
        
        # 4. Считаем loss 
        loss = -torch.mean(log_responsible_outputs * discounted_rewards)
        
        # 5. Считаем градиенты
        self.network.zero_grad()
        loss.backward()
        
        # 6. Сохраняем градиенты в grad_buffer
        if self.grad_buffer is None:
            # Инициализируем буфер 
            self.grad_buffer = []
            for param in self.network.parameters():
                if param.grad is not None:
                    self.grad_buffer.append(param.grad.clone())
                else:
                    self.grad_buffer.append(torch.zeros_like(param))
        else:
            # Накопление градиентов
            for idx, param in enumerate(self.network.parameters()):
                if param.grad is not None:
                    self.grad_buffer[idx] += param.grad
        
        # Очищаем историю (как сброс ep_history)
        self.states = []
        self.actions = []
        self.rewards = []
        self.log_probs = []
    
    def apply_gradients(self):
        """Применяем накопленные градиенты (аналог update_batch)"""
        if self.grad_buffer is not None:
            # Применяем градиенты вручную
            for param, grad in zip(self.network.parameters(), self.grad_buffer):
                if param.grad is None:
                    param.grad = grad.clone()
                else:
                    param.grad += grad
            self.optimizer.step()
            self.optimizer.zero_grad()
            
            self.grad_buffer = None





### Обучение агента

In [ ]:
agent = Agent(lr=1e-2, state_size=4, action_size=2, hidden_size=8)

total_episodes = 5000
max_ep = 999
update_frequency = 5

total_reward = []
total_length = []

for i in range(total_episodes):
    s = env.reset()
    running_reward = 0
    
    for j in range(max_ep):
        # Выбираем действие 
        a = agent.get_action(s)
        
        # Выполняем действие
        s1, r, d, _ = env.step(a)
        
        # Сохраняем награду 
        agent.store_reward(r)
        
        s = s1
        running_reward += r
        
        if d:
            # Обучение на эпизоде
            agent.learn()
            
            if i % update_frequency == 0 and i != 0:
                agent.apply_gradients()
            
            total_reward.append(running_reward)
            total_length.append(j)
            break
    
    if i % 100 == 0:
        print(np.mean(total_reward[-100:]))

env.close()



16.0
21.47
25.57
38.03
43.59
53.05
67.38
90.44
120.19
131.75
162.65
156.48
168.18
181.43
